# genetools quickstart

Post-process a GENE run with the `Run` facade. Every diagnostic is one line away and returns a labelled `xarray.Dataset` plus a `.plot()`.

Set `RUN_DIR` to one of your GENE run directories and run the cells.

In [ ]:
%matplotlib inline
from genetools import Run

RUN_DIR = "/path/to/run"   # <-- edit me
run = Run(RUN_DIR)          # discovers all segments; or Run(RUN_DIR, ext=["_0002", ".dat"])
run

## Metadata

`Run` exposes the run's basic facts and lazily-built pieces.

In [ ]:
print("segments :", run.extensions)
print("species  :", run.species)
print("local    :", run.is_local)
print("n times  :", run.times.size)

## Energy / flux time traces (`nrg`)

In [ ]:
run.nrg.plot()

ds = run.nrg.data            # dims (species, time), vars Q_es, Gamma_es, ...
ds.Q_es.sel(species=run.species[0]).plot()

## Flux spectra (`spectra`)

Auto-dispatches to local (kx/ky) or global spectra based on geometry. `t=(start, stop)` sets the time-average window.

In [ ]:
run.spectra.plot(t=(500, 2000))
ds = run.spectra.data
ds

## Radial profiles and x-resolved fluxes

In [ ]:
run.profiles.plot(t=(1000, 2000))
run.fluxes2d.plot(t=(1000, 2000))

## Linear diagnostics: growth rate and ballooning mode structure

Growth rate / frequency from the field time evolution (γ from |φ| growth, ω from its phase rotation). Ballooning shows the mode structure along the field line for a chosen ky (local runs only).

In [ ]:
run.growthrate.plot()
ky, gamma, omega, window = run.growthrate.compute()
print("ky=", ky, "gamma=", gamma, "omega=", omega)

run.ballooning(ky=0.3).plot()

## Amplitude spectra, zonal flows, shearing rate

In [ ]:
run.amplitude.plot(t=(500, 2000))   # kx/ky |.|^2 spectra of fields & moments
run.zonal.plot()                    # zonal (ky=0) potential x-t contour
run.shearing.plot()                 # ExB shearing rate

## 2-D contours (`contours`)

Real-space (or spectral) 2-D field/moment slices. Plot-only (no `.data`). `field`: 0=phi, 1=A_par, 2=B_par; pass `species=...` for a moment, `ifft` in {None,'x','y','xy'}.

In [ ]:
run.contours.plot(t=(1000, 2000), field=0, ifft="xy")

## Radial transport profiles (`profile_diag`)

Reads GENE's `profile_<species>` **output** (global nonlinear runs): radial T, n, gradients, turbulent (Γ/Q/Π) and neoclassical fluxes, and bootstrap current. Returns an xarray `(species, time, x)` with normalized + `*_SI` variables. `plot()` shows gyro-Bohm by default, **plus** an SI figure when the parameter file has reference units (`si=False`/`si=True` to force one).

In [ ]:
pd = run.profile_diag
ds = pd.data            # xarray (species, time, x): T, n, omt, omn, Gamma, Q, Pi, *_neo, j_boot (+ *_SI)
ds

In [ ]:
pd.plot(t=(1000, 2000))      # gyro-Bohm + SI (if available)

## Equilibrium *input* profiles (`profiles_<species>`)

The background/equilibrium profiles GENE **reads** — distinct from the `profile_<species>` *output* above. Loaded via `EquilibriumProfiles` (not on `Run`).

In [ ]:
from genetools.io import EquilibriumProfiles

eqp = EquilibriumProfiles(RUN_DIR, run.extensions, run.params)
eqp.plot()                       # T, n, R/L_T, R/L_n for all species
ions_bg = eqp[0]["ions"]         # dict: x_o_a, x_o_rho_ref, T, n, omt, omn

## Working with the xarray data

Because diagnostics return `xarray.Dataset`s, slicing, averaging, and plotting are label-based.

In [ ]:
ds = run.fluxes2d.data
ds                                  # inspect dims / coords / units (in .attrs)
# e.g. ds["ions_Qes_x"].plot()      # x-profile of the ion ES heat flux

## Command line

The same diagnostics are available from the shell:

```bash
genetools /path/to/run --nrg
genetools /path/to/run --spectra --t 500 2000 --save spectra.png
genetools /path/to/run --profiles --no-show
genetools /path/to/run --growthrate
genetools /path/to/run --ballooning --ky 0.3
genetools /path/to/run --profile-diag
genetools /path/to/run --contours --field 0 --ifft xy
```